In [ ]:
import numpy as np
from skimage.measure import regionprops

def map_labels_multiple_gt_ior_optimized(predicted, ground_truth, ior_threshold=0.5):
    """
    Maps a single predicted cell to multiple ground truth cells based on
    Intersection over Reference (IoR) exceeding a threshold, optimizing
    by evaluating only overlapping ground truth labels.

    Args:
        predicted (np.ndarray): 3D array of predicted cell labels.
        ground_truth (np.ndarray): 3D array of ground truth cell labels.
        ior_threshold (float): The minimum IoR value for a ground truth
                               cell to be considered a match.

    Returns:
        dict: A dictionary where keys are predicted cell labels. Each value
              is a list of dictionaries, where each dictionary contains the
              'gt_label' and 'ior' for a matching ground truth cell.
    """

    # Get unique labels of predicted cells (excluding background 0)
    predicted_labels = np.unique(predicted[predicted > 0])
    # Initialize an empty dictionary to store the mapping results
    mapping = {}

    # Iterate through each unique predicted cell label
    for pred_label in predicted_labels:
        # Create a boolean mask for the current predicted cell
        predicted_mask = (predicted == pred_label)
        # Find unique ground truth labels that overlap with the current predicted cell
        overlapping_gt_labels = np.unique(ground_truth[predicted_mask])
        # Exclude the background label (0) from the overlapping ground truth labels
        overlapping_gt_labels = overlapping_gt_labels[overlapping_gt_labels > 0]
        # Initialize an empty list to store the matching ground truth labels and their IoR values
        matches = []

        # Iterate through the ground truth labels that have some overlap with the current predicted cell
        for gt_label in overlapping_gt_labels:
            # Create a boolean mask for the current ground truth cell
            ground_truth_mask = (ground_truth == gt_label)

            # Calculate the intersection (number of overlapping voxels)
            intersection = np.logical_and(predicted_mask, ground_truth_mask).sum()
            # Calculate the volume (number of voxels) of the ground truth cell
            ground_truth_volume = ground_truth_mask.sum()

            # Calculate the Intersection over Reference (IoR)
            ior = intersection / ground_truth_volume if ground_truth_volume > 0 else 0.0

            # If the IoR is above the specified threshold, add it to the matches list
            if ior > ior_threshold:
                matches.append({'gt_label': gt_label, 'ior': ior})

        # Store the list of matching ground truth labels and their IoR values for the current predicted cell
        mapping[pred_label] = matches

    return mapping

def map_labels_multiple_gt_ior_regionprops_optimized(predicted, ground_truth, ior_threshold=0.5):
    """
    Maps a single predicted cell to multiple ground truth cells based on
    Intersection over Reference (IoR) exceeding a threshold, optimizing
    by evaluating only overlapping ground truth labels using region properties.

    Args:
        predicted (np.ndarray): 3D array of predicted cell labels.
        ground_truth (np.ndarray): 3D array of ground truth cell labels.
        ior_threshold (float): The minimum IoR value for a ground truth
                               cell to be considered a match.

    Returns:
        dict: A dictionary where keys are predicted cell labels. Each value
              is a list of dictionaries, where each dictionary contains the
              'gt_label' and 'ior' for a matching ground truth cell.
    """

    # Get region properties for each labeled region in the predicted segmentation
    predicted_regions = regionprops(predicted)
    # Get region properties for each labeled region in the ground truth segmentation
    ground_truth_regions = regionprops(ground_truth)
    # Initialize an empty dictionary to store the mapping results
    mapping = {}

    # Create a dictionary to quickly access ground truth regions by their label
    gt_regions_by_label = {region.label: region for region in ground_truth_regions}

    # Iterate through each predicted cell region
    for pred_region in predicted_regions:
        pred_label = pred_region.label
        # Get the set of coordinates (voxels) belonging to the current predicted cell
        pred_coords = set(tuple(coord) for coord in pred_region.coords)

        # Find unique ground truth labels that overlap with the current predicted cell
        overlapping_gt_labels = np.unique(ground_truth[predicted == pred_label])
        # Exclude the background label (0) from the overlapping ground truth labels
        overlapping_gt_labels = overlapping_gt_labels[overlapping_gt_labels > 0]
        # Initialize an empty list to store the matching ground truth labels and their IoR values
        matches = []

        # Iterate through the ground truth labels that have some overlap with the current predicted cell
        for gt_label in overlapping_gt_labels:
            # Check if the ground truth label exists in our pre-computed dictionary
            if gt_label in gt_regions_by_label:
                # Get the region properties for the current ground truth cell
                gt_region = gt_regions_by_label[gt_label]
                # Get the set of coordinates (voxels) belonging to the current ground truth cell
                gt_coords = set(tuple(coord) for coord in gt_region.coords)

                # Calculate the intersection (number of common voxels)
                intersection = len(pred_coords.intersection(gt_coords))
                # Calculate the volume (number of voxels) of the ground truth cell
                ground_truth_volume = len(gt_coords)

                # Calculate the Intersection over Reference (IoR)
                ior = intersection / ground_truth_volume if ground_truth_volume > 0 else 0.0

                # If the IoR is above the specified threshold, add it to the matches list
                if ior > ior_threshold:
                    matches.append({'gt_label': gt_label, 'ior': ior})

        # Store the list of matching ground truth labels and their IoR values for the current predicted cell
        mapping[pred_label] = matches

    return mapping
'''
# Example Usage (same dummy data as before):
predicted_seg = np.zeros((30, 30, 30), dtype=int)
ground_truth_seg = np.zeros((30, 30, 30), dtype=int)
predicted_seg[5:15, 5:15, 5:15] = 1
ground_truth_seg[7:17, 7:17, 5:13] = 5
ground_truth_seg[7:17, 7:17, 13:21] = 6
predicted_seg[20:25, 20:25, 20:25] = 2
ground_truth_seg[22:27, 22:27, 22:27] = 7

print("Label mapping (multiple GT matches, IoR-based, mask-based, optimized):")
label_mapping_multi_gt_ior_optimized_mask = map_labels_multiple_gt_ior_optimized(predicted_seg, ground_truth_seg, ior_threshold=0.3)
print(label_mapping_multi_gt_ior_optimized_mask)

print("\nLabel mapping (multiple GT matches, IoR-based, regionprops-based, optimized):")
label_mapping_multi_gt_ior_optimized_rp = map_labels_multiple_gt_ior_regionprops_optimized(predicted_seg, ground_truth_seg, ior_threshold=0.3)
print(label_mapping_multi_gt_ior_optimized_rp)
'''

In [ ]:
import numpy as np
from skimage.measure import regionprops

def map_labels_with_ior_iou_optimized_with_progress(predicted, ground_truth, ior_threshold=0.5):
    """
    Maps a single predicted cell to multiple ground truth cells if their
    Intersection over Reference (IoR) exceeds a threshold. Calculates and
    saves both IoR and Intersection over Union (IoU) for each match.
    Optimized by evaluating only overlapping ground truth labels and prints progress.

    Args:
        predicted (np.ndarray): 3D array of predicted cell labels.
        ground_truth (np.ndarray): 3D array of ground truth cell labels.
        ior_threshold (float): The minimum IoR value for a ground truth
                               cell to be considered a match.

    Returns:
        dict: A dictionary where keys are predicted cell labels. Each value
              is a list of dictionaries, where each dictionary contains the
              'gt_label', 'ior', and 'iou' for a matching ground truth cell.
    """

    # Get unique labels of predicted cells (excluding background 0)
    predicted_labels = np.unique(predicted[predicted > 0])
    # Get the total number of predicted labels for progress tracking
    total_predicted_labels = len(predicted_labels)
    # Initialize an empty dictionary to store the mapping results
    mapping = {}

    # Iterate through each unique predicted cell label with progress tracking
    for i, pred_label in enumerate(predicted_labels):
        # Create a boolean mask for the current predicted cell
        predicted_mask = (predicted == pred_label)
        # Find unique ground truth labels that overlap with the current predicted cell
        overlapping_gt_labels = np.unique(ground_truth[predicted_mask])
        # Exclude the background label (0) from the overlapping ground truth labels
        overlapping_gt_labels = overlapping_gt_labels[overlapping_gt_labels > 0]
        # Initialize an empty list to store the matching ground truth labels and their IoR and IoU values
        matches = []

        # Iterate through the ground truth labels that have some overlap with the current predicted cell
        for gt_label in overlapping_gt_labels:
            # Create a boolean mask for the current ground truth cell
            ground_truth_mask = (ground_truth == gt_label)

            # Calculate the intersection (number of overlapping voxels)
            intersection = np.logical_and(predicted_mask, ground_truth_mask).sum()
            # Calculate the volume (number of voxels) of the ground truth cell
            ground_truth_volume = ground_truth_mask.sum()
            # Calculate the volume (number of voxels) of the predicted cell
            predicted_volume = predicted_mask.sum()
            # Calculate the volume of the union of the predicted and ground truth cells
            union = predicted_volume + ground_truth_volume - intersection

            # Calculate the Intersection over Reference (IoR)
            ior = intersection / ground_truth_volume if ground_truth_volume > 0 else 0.0
            # Calculate the Intersection over Union (IoU)
            iou = intersection / union if union > 0 else 0.0

            # If the IoR is above the specified threshold, add the match information to the list
            if ior > ior_threshold:
                matches.append({'gt_label': gt_label, 'ior': ior, 'iou': iou})

        # Store the list of matching ground truth labels and their IoR and IoU values for the current predicted cell
        mapping[pred_label] = matches
        # Print the progress of label processing
        print(f"Processed predicted label {i+1}/{total_predicted_labels}", end='\r')

    # Print a completion message
    print("\nLabel mapping complete.")
    return mapping

def map_labels_with_ior_iou_regionprops_optimized_with_progress(predicted, ground_truth, ior_threshold=0.5):
    """
    Maps a single predicted cell to multiple ground truth cells if their
    Intersection over Reference (IoR) exceeds a threshold. Calculates and
    saves both IoR and Intersection over Union (IoU) for each match, using
    region properties. Optimized by evaluating only overlapping ground truth
    labels and prints progress.

    Args:
        predicted (np.ndarray): 3D array of predicted cell labels.
        ground_truth (np.ndarray): 3D array of ground truth cell labels.
        ior_threshold (float): The minimum IoR value for a ground truth
                               cell to be considered a match.

    Returns:
        dict: A dictionary where keys are predicted cell labels. Each value
              is a list of dictionaries, where each dictionary contains the
              'gt_label', 'ior', and 'iou' for a matching ground truth cell.
    """

    # Get region properties for each labeled region in the predicted segmentation
    predicted_regions = regionprops(predicted)
    # Get the total number of predicted regions for progress tracking
    total_predicted_regions = len(predicted_regions)
    # Get region properties for each labeled region in the ground truth segmentation
    ground_truth_regions = regionprops(ground_truth)
    # Initialize an empty dictionary to store the mapping results
    mapping = {}
    # Create a dictionary to quickly access ground truth regions by their label
    gt_regions_by_label = {region.label: region for region in ground_truth_regions}

    # Iterate through each predicted cell region with progress tracking
    for i, pred_region in enumerate(predicted_regions):
        pred_label = pred_region.label
        # Get the set of coordinates (voxels) belonging to the current predicted cell
        pred_coords = set(tuple(coord) for coord in pred_region.coords)
        # Find unique ground truth labels that overlap with the current predicted cell
        overlapping_gt_labels = np.unique(ground_truth[predicted == pred_label])
        # Exclude the background label (0) from the overlapping ground truth labels
        overlapping_gt_labels = overlapping_gt_labels[overlapping_gt_labels > 0]
        # Initialize an empty list to store the matching ground truth labels and their IoR and IoU values
        matches = []

        # Iterate through the ground truth labels that have some overlap with the current predicted cell
        for gt_label in overlapping_gt_labels:
            # Check if the ground truth label exists in our pre-computed dictionary
            if gt_label in gt_regions_by_label:
                # Get the region properties for the current ground truth cell
                gt_region = gt_regions_by_label[gt_label]
                # Get the set of coordinates (voxels) belonging to the current ground truth cell
                gt_coords = set(tuple(coord) for coord in gt_region.coords)

                # Calculate the intersection (number of common voxels)
                intersection = len(pred_coords.intersection(gt_coords))
                # Calculate the volume (number of voxels) of the ground truth cell
                ground_truth_volume = len(gt_coords)
                # Calculate the volume (number of voxels) of the predicted cell
                predicted_volume = len(pred_coords)
                # Calculate the volume of the union of the predicted and ground truth cells
                union = predicted_volume + ground_truth_volume - intersection

                # Calculate the Intersection over Reference (IoR)
                ior = intersection / ground_truth_volume if ground_truth_volume > 0 else 0.0
                # Calculate the Intersection over Union (IoU)
                iou = intersection / union if union > 0 else 0.0

                # If the IoR is above the specified threshold, add the match information to the list
                if ior > ior_threshold:
                    matches.append({'gt_label': gt_label, 'ior': ior, 'iou': iou})

        # Store the list of matching ground truth labels and their IoR and IoU values for the current predicted cell
        mapping[pred_label] = matches
        # Print the progress of label processing
        print(f"Processed predicted label {i+1}/{total_predicted_regions}", end='\r')

    # Print a completion message
    print("\nLabel mapping complete.")
    return mapping

# Example Usage (same dummy data as before):
predicted_seg = np.zeros((30, 30, 30), dtype=int)
ground_truth_seg = np.zeros((30, 30, 30), dtype=int)
predicted_seg[5:15, 5:15, 5:15] = 1
ground_truth_seg[7:17, 7:17, 5:13] = 5
ground_truth_seg[7:17, 7:17, 13:21] = 6
predicted_seg[20:25, 20:25, 20:25] = 2
ground_truth_seg[22:27, 22:27, 22:27] = 7

# Mask-based approach
print("Label mapping (IoR threshold, calculate IoU, mask-based, optimized with progress):")
mapping_results_mask = map_labels_with_ior_iou_optimized_with_progress(predicted_seg, ground_truth_seg, ior_threshold=0.3)
print(mapping_results_mask)

# Regionprops-based approach
print("\nLabel mapping (IoR threshold, calculate IoU, regionprops-based, optimized with progress):")
mapping_results_rp = map_labels_with_ior_iou_regionprops_optimized_with_progress(predicted_seg, ground_truth_seg, ior_threshold=0.3)
print(mapping_results_rp)

def analyze_mapping_results(mapping_results, iou_threshold=0.5, ior_threshold=0.5):
    """
    Analyzes the label mapping results and classifies predicted labels
    as True Positives or Merge Errors based on IoU, IoR, and the number
    of ground truth assignments.

    Args:
        mapping_results (dict): A dictionary where keys are predicted cell
                                labels and values are lists of dictionaries,
                                each containing 'gt_label', 'ior', and 'iou'.
        iou_threshold (float): The minimum IoU value for a match to be
                               considered significant.
        ior_threshold (float): The minimum IoR value for a match to be
                               considered significant.

    Returns:
        dict: A dictionary containing lists of predicted labels classified as
              'true_positives' and 'merge_errors'.
    """
    classifications = {'true_positives': [], 'merge_errors': []}

    for pred_label, matches in mapping_results.items():
        significant_matches = [match for match in matches if match['iou'] > iou_threshold]
        num_significant_matches = len(significant_matches)

        if num_significant_matches == 1 and significant_matches[0]['ior'] > ior_threshold:
            classifications['true_positives'].append(pred_label)
        elif num_significant_matches > 1:
            # Check if IoR for these multiple matches is generally below the IoR threshold
            all_ior_below_threshold = all(match['ior'] <= ior_threshold for match in significant_matches)
            if all_ior_below_threshold:
                classifications['merge_errors'].append(pred_label)

    return classifications

In [ ]:
import tifffile
import numpy as np
gt_path = '/content/drive/MyDrive/Colab Notebooks/data/GT/02262025_atp_dapi_488_555_647_ovary1_ch1_downs_cp_masks.tif'
pred_path = '/content/drive/MyDrive/Colab Notebooks/data/segmentation/02262025_atp_dapi_488_555_647_ovary1_ch1_downs_cp_masks.tif'

gt_data =np.array(tifffile.imread(gt_path))
pred_data =np.array(tifffile.imread(pred_path))

ior_threshold = 0.5


# Run the mapping function (using the mask-based one for this example)
mapping_results = map_labels_with_ior_iou_optimized_with_progress(pred_data, gt_data, ior_threshold=0.2)

iou_threshold = 0.8
ior_threshold = 0.8

# Analyze the results
classification_results = analyze_mapping_results(mapping_results, iou_threshold, ior_threshold)

print("\nClassification Results:")
print(f"True Positives: {classification_results['true_positives']}")
print(f"Merge Errors: {classification_results['merge_errors']}")

In [ ]:

def analyze_merge_errors(mapping_results):
    """
    Analyzes the label mapping results to identify potential merge errors
    by counting the number of ground truth labels assigned to each
    predicted label.

    Args:
        mapping_results (dict): A dictionary where keys are predicted cell
                                labels and values are lists of dictionaries,
                                each containing 'gt_label' and 'ior'.

    Returns:
        dict: A dictionary where keys are predicted cell labels and values
              are the number of ground truth labels assigned to them
              (i.e., the length of the list of matches).
    """
    merge_error_analysis = {}
    for pred_label, matches in mapping_results.items():
        merge_error_analysis[pred_label] = len(matches)
    return merge_error_analysis



print("\nMerge Error Analysis (Regionprops-Based):")
merge_errors_rp = analyze_merge_errors(mapping_results)
for pred_label, num_assigned_gt in merge_errors_rp.items():
    print(f"Predicted Label {pred_label}: Assigned to {num_assigned_gt} GT labels")

In [1]:
import numpy as np
import pandas as pd
import time # Optional: for timing the execution

def evaluate_3d_segmentation(pred_labels, gt_labels, ior_threshold=0.5, iou_threshold=0.5):
    """
    Evaluates a 3D segmentation based on IoR matching and classifies detections.

    Maps predicted labels to ground truth labels using Intersection over
    Reference (IoR) > ior_threshold. Allows multiple GT labels to be mapped
    to one predicted label (Merge Error detection). Calculates IoR and IoU
    for matched pairs. Classifies predicted labels as:
        - TP (True Positive): Matches one GT label with IoR > ior_threshold
                              AND IoU > iou_threshold.
        - FP (False Positive): No GT label match with IoR > ior_threshold.
        - MergeError: Matches more than one GT label with IoR > ior_threshold.
        - PoorOverlap: Matches one GT label with IoR > ior_threshold but
                       IoU <= iou_threshold.

    Args:
        pred_labels (np.ndarray): 3D array with predicted integer labels (0=background).
        gt_labels (np.ndarray): 3D array with ground truth integer labels (0=background).
        ior_threshold (float): Minimum IoR for a match (localization criterion). Defaults to 0.5.
        iou_threshold (float): Minimum IoU for a TP classification (overlap quality). Defaults to 0.5.

    Returns:
        tuple:
            - pd.DataFrame: Evaluation results with columns
                          ['pred_label', 'gt_label', 'IoR', 'IoU', 'type'].
                          'gt_label' is None for FP. For MergeError, 'pred_label'
                          will have multiple rows, one for each merged 'gt_label'.
            - set: Set of ground truth label IDs that were not matched by any
                   prediction with IoR > ior_threshold (False Negatives).
    """
    start_time = time.time()

    # --- Input Validation ---
    if not isinstance(pred_labels, np.ndarray) or not isinstance(gt_labels, np.ndarray):
        raise TypeError("Inputs must be numpy arrays.")
    if pred_labels.shape != gt_labels.shape:
        raise ValueError("Prediction and Ground Truth must have the same shape.")
    if pred_labels.dtype.kind not in 'iu' or gt_labels.dtype.kind not in 'iu':
         print("Warning: Labels arrays are not integer types. Ensure labels are discrete integers.")


    print(f"Evaluating segmentation for shape: {pred_labels.shape}")
    print(f"IoR threshold: {ior_threshold}, IoU threshold: {iou_threshold}")

    # --- Get Unique Labels (excluding background 0) ---
    pred_ids = np.unique(pred_labels)
    pred_ids = pred_ids[pred_ids != 0]
    gt_ids = np.unique(gt_labels)
    gt_ids = gt_ids[gt_ids != 0]

    print(f"Found {len(pred_ids)} predicted objects and {len(gt_ids)} ground truth objects.")

    if len(pred_ids) == 0 or len(gt_ids) == 0:
        print("Warning: No predicted objects or no ground truth objects found.")
        # Return empty results if either set is empty
        return pd.DataFrame(columns=['pred_label', 'gt_label', 'IoR', 'IoU', 'type']), set(gt_ids)


    # --- Pre-calculate Volumes (voxel counts) for efficiency ---
    print("Calculating label volumes...")
    volumes_pred = {pid: np.sum(pred_labels == pid) for pid in pred_ids}
    volumes_gt = {gid: np.sum(gt_labels == gid) for gid in gt_ids}

    results = []
    matched_gt_ids = set() # Keep track of GT IDs that have been matched by at least one pred

    # --- Iterate through each Predicted Label ---
    print(f"Matching predicted labels to ground truth...")
    for i, pred_id in enumerate(pred_ids):
        if (i + 1) % max(1, len(pred_ids) // 10) == 0: # Print progress
             print(f"  Processing predicted label {i+1}/{len(pred_ids)}...")

        pred_mask = (pred_labels == pred_id)
        pred_volume = volumes_pred[pred_id]

        # --- Optimization: Find Overlapping GT labels ---
        # Get GT labels present *only* within the bounding box of the pred_mask is faster
        # but less precise than checking the actual mask voxels. Checking mask voxels is safer.
        overlapping_gt_ids_in_mask = np.unique(gt_labels[pred_mask])
        # Exclude background label 0
        overlapping_gt_ids = overlapping_gt_ids_in_mask[overlapping_gt_ids_in_mask != 0]

        current_matches = [] # Store {'gt_id': id, 'ior': ior_val, 'iou': iou_val} for this pred_id

        if len(overlapping_gt_ids) > 0:
            # --- Iterate only through GT labels that spatially overlap ---
            for gt_id in overlapping_gt_ids:
                 # Check if gt_id is valid (might have been filtered out if empty, though unlikely here)
                 if gt_id not in volumes_gt:
                     continue

                 gt_mask = (gt_labels == gt_id)
                 gt_volume = volumes_gt[gt_id]

                 # Calculate Intersection
                 intersection = np.sum(pred_mask & gt_mask)

                 if intersection == 0: # Should not happen with overlapping_gt_ids, but check
                      continue

                 # Calculate IoR (Intersection over Reference/GT)
                 # Avoid division by zero if gt_volume is somehow 0
                 ior = intersection / gt_volume if gt_volume > 0 else 0

                 # --- Check IoR Match Criterion ---
                 if ior > ior_threshold:
                     # Calculate Union and IoU
                     union = pred_volume + gt_volume - intersection
                     # Avoid division by zero if union is 0
                     iou = intersection / union if union > 0 else 0

                     current_matches.append({'gt_id': gt_id, 'ior': ior, 'iou': iou})
                     # Mark GT as potentially matched (will be confirmed below)


        # --- Classify the Predicted Label based on matches ---
        num_matches = len(current_matches)

        if num_matches == 0:
            # No GT label matched with IoR > threshold -> False Positive
            results.append({
                'pred_label': pred_id,
                'gt_label': None, # Explicitly None for FPs
                'IoR': 0.0,
                'IoU': 0.0,
                'type': 'FP'
            })
        elif num_matches == 1:
            # Exactly one GT label matched -> check IoU for TP or PoorOverlap
            match = current_matches[0]
            gt_id_matched = match['gt_id']
            matched_gt_ids.add(gt_id_matched) # Confirm match for FN tracking

            if match['iou'] > iou_threshold:
                match_type = 'TP' # True Positive
            else:
                match_type = 'PoorOverlap' # Matched by IoR, but low IoU

            results.append({
                'pred_label': pred_id,
                'gt_label': gt_id_matched,
                'IoR': match['ior'],
                'IoU': match['iou'],
                'type': match_type
            })
        else:
            # Multiple GT labels matched (IoR > threshold for >1 GT) -> Merge Error
            # Add a row for *each* GT object involved in the merge
            for match in current_matches:
                 gt_id_merged = match['gt_id']
                 matched_gt_ids.add(gt_id_merged) # Confirm match for FN tracking
                 results.append({
                    'pred_label': pred_id,     # Same predicted label for all merged rows
                    'gt_label': gt_id_merged,  # The specific GT ID involved in this row
                    'IoR': match['ior'],       # IoR specific to this pred-GT pair
                    'IoU': match['iou'],       # IoU specific to this pred-GT pair
                    'type': 'MergeError'       # Classification for the predicted label
                 })

    # --- Identify False Negatives ---
    all_gt_ids_set = set(gt_ids)
    fn_gt_ids = all_gt_ids_set - matched_gt_ids
    print(f"Identified {len(fn_gt_ids)} False Negative GT labels.")

    # --- Create DataFrame ---
    print("Creating results DataFrame...")
    results_df = pd.DataFrame(results)

    # Reorder columns for clarity if the DataFrame is not empty
    if not results_df.empty:
        results_df = results_df[['pred_label', 'gt_label', 'IoR', 'IoU', 'type']]
        # Ensure correct types for columns where None might appear
        results_df['gt_label'] = results_df['gt_label'].astype(object) # To allow None or int
    else:
        # Handle case with no predictions or no matches -> return empty DF with correct columns
         results_df = pd.DataFrame(columns=['pred_label', 'gt_label', 'IoR', 'IoU', 'type'])


    end_time = time.time()
    print(f"Evaluation completed in {end_time - start_time:.2f} seconds.")

    return results_df, fn_gt_ids



In [ ]:
import tifffile
import numpy as np
gt_path = 'D:\Projects\Mikala\GT\corrected_GT/GT_02222025_control_dapi_488_555_647_ovary1_corrected.tif'
pred_path = 'D:\Projects\Mikala\GT\eval/02222025_control_dapi_488_555_647_ovary1_ch1_downs_cp_masks.tif'

gt_labels =np.array(tifffile.imread(gt_path))
pred_labels =np.array(tifffile.imread(pred_path))

ior_threshold = 0.5
iou_threshold = 0.5




print("----- Running Evaluation -----")
evaluation_df, false_negatives = evaluate_3d_segmentation(pred_labels, gt_labels, ior_threshold, iou_threshold)

print("\n----- Evaluation Results (DataFrame) -----")
# Use to_string() to prevent truncation in output
print(evaluation_df.to_string())

print("\n----- False Negative GT Labels -----")
print(sorted(list(false_negatives))) # Sort for consistent output

# --- You can further analyze the DataFrame ---
print("\n----- Summary -----")
if not evaluation_df.empty:
    counts = evaluation_df['type'].value_counts()
    # Ensure all types are present in the output, even if count is 0
    all_types = ['TP', 'FP', 'MergeError', 'PoorOverlap']
    summary_counts = {t: counts.get(t, 0) for t in all_types}

    # Note: MergeError count in the summary might be misleading if you want # of merge events
    # vs # of GTs involved in merges. The DataFrame shows the latter.
    # Count unique pred_labels with type 'MergeError' for number of merge events:
    merge_events = evaluation_df[evaluation_df['type'] == 'MergeError']['pred_label'].nunique()

    print(f"True Positives (TP):       {summary_counts['TP']}")
    print(f"False Positives (FP):      {summary_counts['FP']}")
    print(f"Poor Overlap Matches:    {summary_counts['PoorOverlap']}")
    print(f"Merge Errors (Events):   {merge_events}")
    print(f"  (GT labels involved):  {summary_counts['MergeError']}")
    print(f"False Negatives (FN):    {len(false_negatives)}")
else:
    print("DataFrame is empty, no summary generated.")

In [ ]:
import numpy as np
from collections import defaultdict

import tifffile
import numpy as np
import os

def evaluate_segmentation_3d(
    gt_labels: np.ndarray,
    pred_labels: np.ndarray,
    iou_threshold: float = 0.5,
    ior_threshold: float = 0.5
) -> dict:
    """
    Evaluates 3D instance segmentation performance, calculating TP, FP, FN,
    and specifically identifying merge and split errors.

    The logic follows these steps:
    1.  Find all overlapping pairs of ground truth (GT) and predicted labels.
    2.  Calculate IoU (Intersection over Union) and IoR (Intersection over Reference,
        i.e., over GT area) for each pair.
    3.  Map predictions to GT labels if IoR > ior_threshold. This allows a
        single prediction to match multiple GT labels (a potential merge).
    4.  Identify and classify all matches (one-to-one, merge, split) and errors.
    5.  Compile a detailed DataFrame of each event.
    6.  Return both a summary dictionary and the detailed DataFrame.

    Args:
        gt_labels (np.ndarray): A 3D labeled image for the ground truth.
        pred_labels (np.ndarray): A 3D labeled image for the prediction.
        iou_threshold (float): IoU threshold to be considered a True Positive.
        ior_threshold (float): IoR threshold to establish a potential match
                               between a prediction and a ground truth object.

    Returns:
        dict: A dictionary containing:
              'summary': A dict with the overall metrics ('tp', 'fp', 'fn', etc.).
              'details': A pandas DataFrame with a detailed breakdown of each object.
    """
    # Get unique labels from GT and prediction, ignoring background (label 0)
    gt_label_ids = np.unique(gt_labels[gt_labels > 0])
    pred_label_ids = np.unique(pred_labels[pred_labels > 0])

    # --- Step 1: Pre-calculate all pairwise metrics for efficiency ---
    pairwise_metrics = []
    gt_masks = {label_id: (gt_labels == label_id) for label_id in gt_label_ids}
    pred_masks = {label_id: (pred_labels == label_id) for label_id in pred_label_ids}

    for p_id in pred_label_ids:
        pred_mask = pred_masks[p_id]
        overlapping_gt_ids = np.unique(gt_labels[pred_mask])
        for gt_id in overlapping_gt_ids:
            if gt_id == 0: continue
            gt_mask = gt_masks[gt_id]
            intersection = np.sum(pred_mask & gt_mask)
            union = np.sum(pred_mask | gt_mask)
            gt_area = np.sum(gt_mask)
            iou = intersection / union if union > 0 else 0
            ior = intersection / gt_area if gt_area > 0 else 0
            pairwise_metrics.append({'p_id': p_id, 'gt_id': gt_id, 'iou': iou, 'ior': ior})

    # --- Step 2: Establish Mappings based on IoR threshold ---
    pred_to_gt_map = defaultdict(list)
    gt_to_pred_map = defaultdict(list)
    for metrics in pairwise_metrics:
        if metrics['ior'] > ior_threshold:
            pred_to_gt_map[metrics['p_id']].append(metrics)
    for p_id, gt_list in pred_to_gt_map.items():
        for gt_info in gt_list:
            gt_to_pred_map[gt_info['gt_id']].append(gt_info)

    # --- Step 3: Classify matches and build detailed results list ---
    tp, fp, fn = 0, 0, 0
    merge_errors, split_errors = 0, 0
    processed_preds, processed_gts = set(), set()
    results_list = []

    # Identify and process MERGE errors (1 pred -> N GTs)
    for p_id, gt_list in pred_to_gt_map.items():
        if len(gt_list) > 1:
            merge_errors += 1
            processed_preds.add(p_id)
            gt_list.sort(key=lambda x: x['iou'], reverse=True)
            best_gt_match = gt_list[0]
            
            # Log the primary interaction for the merge error
            results_list.append({
                'pred_label': p_id, 'gt_label': best_gt_match['gt_id'],
                'IoR': best_gt_match['ior'], 'IoU': best_gt_match['iou'], 'type': 'merge_error'
            })
            
            if best_gt_match['iou'] > iou_threshold:
                tp += 1
            else:
                fp += 1
                fn += 1
            processed_gts.add(best_gt_match['gt_id'])
            
            for other_gt in gt_list[1:]:
                fn += 1
                processed_gts.add(other_gt['gt_id'])
                # Log the other GTs involved in the merge as FNs
                results_list.append({
                    'pred_label': None, 'gt_label': other_gt['gt_id'],
                    'IoR': np.nan, 'IoU': np.nan, 'type': 'fn'
                })

    # Identify and process SPLIT errors (M preds -> 1 GT)
    for gt_id, p_list in gt_to_pred_map.items():
        if gt_id in processed_gts or len(p_list) <= 1: continue
        split_errors += 1
        processed_gts.add(gt_id)
        p_list.sort(key=lambda x: x['iou'], reverse=True)
        best_p_match = p_list[0]
        
        results_list.append({
            'pred_label': best_p_match['p_id'], 'gt_label': gt_id,
            'IoR': best_p_match['ior'], 'IoU': best_p_match['iou'], 'type': 'split_error'
        })
        
        if best_p_match['iou'] > iou_threshold:
            tp += 1
        else:
            fp += 1; fn += 1
        processed_preds.add(best_p_match['p_id'])
        
        for other_p in p_list[1:]:
            fp += 1
            processed_preds.add(other_p['p_id'])
            results_list.append({
                'pred_label': other_p['p_id'], 'gt_label': None,
                'IoR': np.nan, 'IoU': np.nan, 'type': 'fp'
            })

    # Classify clean one-to-one matches
    for p_id, gt_list in pred_to_gt_map.items():
        if p_id in processed_preds: continue
        gt_id = gt_list[0]['gt_id']
        if gt_id in processed_gts: continue
        
        iou = gt_list[0]['iou']
        ior = gt_list[0]['ior']
        
        if iou > iou_threshold:
            tp += 1
            results_list.append({
                'pred_label': p_id, 'gt_label': gt_id, 'IoR': ior, 'IoU': iou, 'type': 'tp'
            })
        else:
            fp += 1; fn += 1
            # A low IoU match counts as both an FP and an FN. We log them as two events in the table.
            results_list.append({
                'pred_label': p_id, 'gt_label': gt_id, 'IoR': ior, 'IoU': iou, 'type': 'fp'
            })
            results_list.append({
                'pred_label': None, 'gt_label': gt_id, 'IoR': np.nan, 'IoU': np.nan, 'type': 'fn'
            })
        processed_preds.add(p_id)
        processed_gts.add(gt_id)

    # --- Step 4: Log remaining unmatched objects ---
    unmatched_preds = set(pred_label_ids) - processed_preds
    fp += len(unmatched_preds)
    for p_id in unmatched_preds:
        results_list.append({
            'pred_label': p_id, 'gt_label': None, 'IoR': np.nan, 'IoU': np.nan, 'type': 'fp'
        })

    unmatched_gts = set(gt_label_ids) - processed_gts
    fn += len(unmatched_gts)
    for gt_id in unmatched_gts:
        results_list.append({
            'pred_label': None, 'gt_label': gt_id, 'IoR': np.nan, 'IoU': np.nan, 'type': 'fn'
        })

    # --- Step 5: Final Assembly ---
    summary_dict = {'tp': tp, 'fp': fp, 'fn': fn, 'merge_errors': merge_errors, 'split_errors': split_errors}
    details_df = pd.DataFrame(results_list, columns=['pred_label', 'gt_label', 'IoR', 'IoU', 'type'])
    # Convert nullable integer columns
    details_df['pred_label'] = details_df['pred_label'].astype('Int64')
    details_df['gt_label'] = details_df['gt_label'].astype('Int64')
    
    return {'summary': summary_dict, 'details': details_df}





In [ ]:
import os
import numpy as np
import pandas as pd
import tifffile
gt_path = 'D:\Mikala\images\GT\corrected_GT'
pred_path = 'D:\\Mikala\\images\\tunning\\cp4_stitch03'

pred_fileList = [f for f in os.listdir(pred_path)  if f.endswith(('.tif','.tiff'))]  
gt_fileList = [os.path.splitext(f)[0]+'_corrected.tif' for f in pred_fileList ] 


qc_details = []
qc_summary = []

for idx,f in enumerate(gt_fileList):
    print('\n[processFiles] processing file ' + str(idx+1) + ' of ' +str(len(gt_fileList)))
    print('[processFiles] Filename = ' + f)

    gt_labels =np.array(tifffile.imread(os.path.join(gt_path,f)))
    pred_labels =np.array(tifffile.imread(os.path.join(pred_path,pred_fileList[idx])))
    
    ior_threshold = 0.5
    iou_threshold = 0.5

    evaluation = evaluate_segmentation_3d(gt_labels, pred_labels, iou_threshold, ior_threshold)
    summary = pd.DataFrame(evaluation['summary'], index = [0] ).reset_index(drop=True)
    details_df = evaluation['details']

    PPV = summary['tp']/(summary['tp'] + summary['fp']) #Precision
    Sensitivity = summary['tp'][0]/(summary['tp'][0] + summary['fn'][0]) #recall
    F1 = (2*PPV*Sensitivity) /(PPV + Sensitivity)
    merge_errors = summary['merge_errors'][0]/len(pd.unique(details_df['gt_label']))
    split_errors = summary['split_errors'][0]/len(pd.unique(details_df['gt_label']))

    summary.insert(loc=0, column = 'total_cells_gt', value = len(pd.unique(details_df['gt_label'])))
    summary.insert(loc=0, column = 'split_error_rate', value = split_errors )
    summary.insert(loc=0, column = 'merge_error_rate', value = merge_errors )

    summary.insert(loc=0, column = 'PPV', value = PPV )
    summary.insert(loc=0, column = 'Sensitivity', value = Sensitivity)
    summary.insert(loc=0, column = 'F1', value = F1)

    summary.insert(loc=0, column = 'file', value = f)

    
    

  
    details_df.insert(loc=0, column = 'file', value = f)

    qc_details.append(details_df)
    qc_summary.append(summary)

qc_details = pd.concat(qc_details, ignore_index=True)
qc_summary = pd.concat(qc_summary, ignore_index=True)
qc_total_summary = qc_summary.sum(axis=0) 

PPV = qc_total_summary['tp']/(qc_total_summary['tp'] + qc_total_summary['fp']) #Precision
Sensitivity = qc_total_summary['tp']/(qc_total_summary['tp'] + qc_total_summary['fn']) #recall

F1 = (2*PPV*Sensitivity) /(PPV + Sensitivity)

print ('Precision: ' + str(PPV))
print ('Sensitivity: '+ str(Sensitivity))
print ('F1 Score: ', str(F1))
print('Merge  error rate: ',  qc_total_summary['merge_errors']/qc_total_summary['total_cells_gt'])
print('Split error rate: ',  qc_total_summary['split_errors']/qc_total_summary['total_cells_gt'])
        

In [ ]:
import os
import numpy as np
import pandas as pd
import tifffile
gt_path = 'D:\Mikala\images\GT\corrected_GT'
pred_path = 'D:\\Mikala\\images\\tunning\\cp4_stitch03'

pred_fileList = [f for f in os.listdir(pred_path)  if f.endswith(('.tif','.tiff'))]  
gt_fileList = [os.path.splitext(f)[0]+'_corrected.tif' for f in pred_fileList ] 


qc_details = []
qc_summary = []

for idx,f in enumerate(gt_fileList):
    print('\n[processFiles] processing file ' + str(idx+1) + ' of ' +str(len(gt_fileList)))
    print('[processFiles] Filename = ' + f)

    gt_labels =np.array(tifffile.imread(os.path.join(gt_path,f)))
    pred_labels =np.array(tifffile.imread(os.path.join(pred_path,pred_fileList[idx])))
    
    ior_threshold = 0.5
    iou_threshold = 0.5

    evaluation = evaluate_segmentation_3d(gt_labels, pred_labels, iou_threshold, ior_threshold)
    summary = pd.DataFrame(evaluation['summary'], index = [0] ).reset_index(drop=True)
    details_df = evaluation['details']

    PPV = summary['tp']/(summary['tp'] + summary['fp']) #Precision
    Sensitivity = summary['tp'][0]/(summary['tp'][0] + summary['fn'][0]) #recall
    F1 = (2*PPV*Sensitivity) /(PPV + Sensitivity)
    merge_errors = summary['merge_errors'][0]/len(pd.unique(details_df['gt_label']))
    split_errors = summary['split_errors'][0]/len(pd.unique(details_df['gt_label']))

    summary.insert(loc=0, column = 'total_cells_gt', value = len(pd.unique(details_df['gt_label'])))
    summary.insert(loc=0, column = 'split_error_rate', value = split_errors )
    summary.insert(loc=0, column = 'merge_error_rate', value = merge_errors )

    summary.insert(loc=0, column = 'PPV', value = PPV )
    summary.insert(loc=0, column = 'Sensitivity', value = Sensitivity)
    summary.insert(loc=0, column = 'F1', value = F1)

    summary.insert(loc=0, column = 'file', value = f)

    
    

  
    details_df.insert(loc=0, column = 'file', value = f)

    qc_details.append(details_df)
    qc_summary.append(summary)

qc_details = pd.concat(qc_details, ignore_index=True)
qc_summary = pd.concat(qc_summary, ignore_index=True)
qc_total_summary = qc_summary.sum(axis=0) 

PPV = qc_total_summary['tp']/(qc_total_summary['tp'] + qc_total_summary['fp']) #Precision
Sensitivity = qc_total_summary['tp']/(qc_total_summary['tp'] + qc_total_summary['fn']) #recall

F1 = (2*PPV*Sensitivity) /(PPV + Sensitivity)

print ('Precision: ' + str(PPV))
print ('Sensitivity: '+ str(Sensitivity))
print ('F1 Score: ', str(F1))
print('Merge  error rate: ',  qc_total_summary['merge_errors']/qc_total_summary['total_cells_gt'])
print('Split error rate: ',  qc_total_summary['split_errors']/qc_total_summary['total_cells_gt'])
        